8) LCEL 체인에 메모리 추가하기

In [ ]:
# [목적] LCEL 체인에서 대화 메모리를 사용할 구성 요소를 준비하는 예제
# 프롬프트, 메모리, 실행 가능한 Runnable, 채팅 모델을 불러와 이후 파이프라인을 만들 준비를 합니다.
# 대화 이력을 자동으로 입력에 포함하는 체인을 구성하기 위한 첫 단계입니다.
from operator import itemgetter
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI()

In [ ]:
# [목적] 시스템 지시문·대화 이력·새 질문을 조합하는 채팅 프롬프트를 만드는 예제
# MessagesPlaceholder가 실행 시 전달되는 chat_history를 메시지 목록 사이에 삽입합니다.
# 모델이 이전 대화를 참고하면서 현재 질문에 답하도록 입력 구조를 정합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
# [목적] 메시지 목록 형태의 대화 이력을 저장할 ConversationBufferMemory를 만드는 예제
# return_messages=True로 역할이 구분된 메시지를 보관하고, memory_key로 프롬프트의 변수 이름을 맞춥니다.
# 저장된 이력은 Runnable을 통해 다음 모델 호출의 chat_history에 전달됩니다.
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history"
)

In [ ]:
# [목적] 새 메모리가 반환하는 초기 대화 이력 구조를 확인하는 예제
# 아직 저장된 문답이 없으므로 memory_key에 대응하는 빈 메시지 목록이 반환됩니다.
# 이후 Runnable이 참조할 메모리 값의 형태를 미리 점검합니다.
memory.load_memory_variables({})  # 메모리 변수를 빈 딕셔너리로 초기화

In [ ]:
# [목적] 입력에 메모리의 chat_history를 자동으로 추가하는 Runnable을 구성하는 예제
# assign가 기존 input은 유지하고 메모리 조회 결과에서 chat_history 값만 골라 새 필드로 넣습니다.
# 프롬프트가 요구하는 대화 이력을 매 호출마다 준비하기 위해 사용합니다.
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")  # memory_key와 동일하게 입력
)

runnable.invoke({"input": "hi"})

In [ ]:
# [목적] 메모리 조회 기능을 독립적인 Runnable로 정의하는 예제
# RunnableLambda는 일반 메서드인 load_memory_variables를 체인에서 연결 가능한 단계로 바꿉니다.
# 이후 입력을 받아 대화 이력을 반환하는 재사용 가능한 구성 요소로 활용할 수 있습니다.
chat_history = RunnableLambda(memory.load_memory_variables)

In [ ]:
# [목적] Runnable 파이프라인으로 입력과 대화 이력을 함께 전달하는 방식을 다시 확인하는 예제
# 새 입력에 메모리에서 꺼낸 chat_history를 붙인 뒤 invoke로 실제 전달 값을 확인합니다.
# 체인 연결 전에 데이터가 프롬프트 형식에 맞게 준비되는지 검증합니다.
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")
)

runnable.invoke({"input": "hi"})

In [ ]:
# [목적] 메모리 처리·프롬프트·채팅 모델을 하나의 LCEL 체인으로 연결하는 예제
# | 연산자는 앞 단계의 결과를 다음 단계 입력으로 전달해 runnable, prompt, model을 순서대로 실행합니다.
# 이 체인은 새 질문에 대화 이력을 함께 넣어 모델 응답을 생성합니다.
chain = runnable | prompt | model

In [ ]:
# [목적] 대화 이력 자리표시자를 포함한 채팅 프롬프트를 다시 정의하는 예제
# 시스템 역할, 이전 메시지, 현재 사용자 입력을 정해진 순서로 구성합니다.
# 메모리가 포함된 입력을 모델에 전달할 프롬프트 구조를 명확히 확인합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
# [목적] 첫 사용자 인사를 체인에 전달해 모델 응답을 생성하는 예제
# chain.invoke가 입력에 현재 메모리 이력을 보태 프롬프트를 만들고 채팅 모델을 호출합니다.
# 생성된 응답은 다음 셀에서 메모리에 저장해 후속 대화의 문맥으로 사용합니다.
response = chain.invoke(
    {"input": "만나서 반갑습니다. 제 이름은 승훈입니다."}
)

print(response.content)  # 생성된 응답을 출력

In [ ]:
# [목적] 메모리 연결 Runnable이 현재 입력에 이력을 추가하는 결과를 다시 확인하는 예제
# invoke 결과에서 input과 chat_history가 함께 구성되는지 확인합니다.
# 모델 호출 전 메모리 전달 단계가 정상인지 점검하는 용도입니다.
runnable.invoke({"input": "hi"})

In [ ]:
# [목적] 첫 문답을 메모리에 저장하고 갱신된 대화 이력을 확인하는 예제
# save_context에 사용자 입력과 모델 응답을 넣으면 다음 호출에서 참조할 메시지 목록에 추가됩니다.
# 저장 후 조회하여 대화 문맥이 실제로 누적되었는지 확인합니다.
memory.save_context(  # 입력된 데이터와 응답 내용을 메모리에 저장
    {"human": "만나서 반갑습니다. 제 이름은 승훈입니다."},
    {"ai": response.content}
)

memory.load_memory_variables({})  # 저장된 대화 기록을 출력

In [ ]:
# [목적] 저장된 이전 인사를 바탕으로 이름을 묻는 후속 질문에 답하는 예제
# 체인이 메모리의 chat_history를 함께 전달하므로 모델은 앞서 소개된 이름을 참고할 수 있습니다.
# 대화 메모리가 문맥을 유지하는 최종 결과를 확인합니다.
response = chain.invoke(
    {"input": "제 이름이 무엇이었는지 기억하세요?"}
)

print(response.content)